In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("loan_approval_data.csv")

In [ ]:
df.head()
df.info()
df.isnull().sum()
df.describe()

# Handle Missing Values

In [ ]:
categorical_cols = df.select_dtypes(include=["str"]).columns
numerical_cols = df.select_dtypes(include=["number"]).columns

In [ ]:
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols]) 

In [ ]:
df.head()
df.isnull().sum()

In [ ]:
cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols]) 

# EDA - exploratory data analysis

In [ ]:
# how balanced our classes are?

classes_count = df["Loan_Approved"].value_counts()

plt.pie(classes_count, labels=["No", "Yes"], autopct="%1.1f%%")
plt.title("Is Loan approved or not?")

In [ ]:
# analyze categories

# gender_cnt = df["Gender"].value_counts()
# ax = sns.barplot(gender_cnt)
# ax.bar_label(ax.containers[0])

# edu_cnt = df["Education_Level"].value_counts()
# ax = sns.barplot(edu_cnt)
# ax.bar_label(ax.containers[0])

# mar_cnt = df["Marital_Status"].value_counts()
# ax = sns.barplot(mar_cnt)
# ax.bar_label(ax.containers[0])

pur_cnt = df["Loan_Purpose"].value_counts()
ax = sns.barplot(pur_cnt)
ax.bar_label(ax.containers[0])

In [ ]:
# analyze income
sns.histplot(
    data=df,
    x="Applicant_Income",
    bins=20
)

In [ ]:
sns.histplot(
    data=df,
    x="Coapplicant_Income",
    bins=20
)

In [ ]:
# outliers - boxplot

sns.boxplot(
    data=df,
    x="Loan_Approved",
    y="Applicant_Income"
)

In [ ]:
fig, axes = plt.subplots(3, 2)
sns.boxplot(ax=axes[0][0], data=df, x="Loan_Approved", y="Applicant_Income")
sns.boxplot(ax=axes[0][1], data=df, x="Loan_Approved", y="Credit_Score")
sns.boxplot(ax=axes[1][0], data=df, x="Loan_Approved", y="DTI_Ratio")
sns.boxplot(ax=axes[1][1], data=df, x="Loan_Approved", y="Savings")
sns.boxplot(ax=axes[2][0], data=df, x="Loan_Approved", y="Age")
sns.boxplot(ax=axes[2][1], data=df, x="Loan_Approved", y="Loan_Amount")
plt.tight_layout()

In [ ]:
# Credit Score with Loan Approved

sns.histplot(
    data=df,
    x="Credit_Score",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
)

In [ ]:
# Remove Applicant Id
df = df.drop("Applicant_ID", axis=1)

In [ ]:
#df.head()
df.columns
df.info()

# Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
le = LabelEncoder()
df["Education_Level"] = le.fit_transform(df["Education_Level"])
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])
  

In [ ]:
df.head()

In [ ]:
cols = ["Employment_Status", "Marital_Status", "Loan_Purpose", "Property_Area", "Gender", "Employer_Category"]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")

encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=df.index)
df = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

In [ ]:
df.head()
df.describe()
df.info()

# Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include="number")
corr_matrix = num_cols.corr()

plt.figure(figsize=((15,8)))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
    

In [ ]:
num_cols.corr()["Loan_Approved"].sort_values(ascending=False)

# Train-Test-Split + Feature Scaling

In [ ]:
X = df.drop("Loan_Approved", axis=1)
y = df["Loan_Approved"]

In [ ]:
y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_test.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled

# Train & Evaluate Models

In [ ]:
# Logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation
print("Logistic Regression Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion matrix: ", confusion_matrix(y_test, y_pred))

In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation
print("KNN Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion matrix: ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation
print("Naive Bayes Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion matrix: ", confusion_matrix(y_test, y_pred))

# Best Model on the basis of Precision => Naive Bayes

# Feature Engineering

In [ ]:
# Add or transform features
df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2


# Skewed features
#df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

X = df.drop(columns = ["Loan_Approved", "DTI_Ratio", "Credit_Score"])
y = df["Loan_Approved"]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train.head()

In [ ]:
# Logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation
print("Logistic Regression Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion matrix: ", confusion_matrix(y_test, y_pred))

In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation
print("KNN Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion matrix: ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation
print("Naive Bayes Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion matrix: ", confusion_matrix(y_test, y_pred))

In [ ]:
import os
os.makedirs("images", exist_ok=True)

models = ["Logistic Regression", "KNN", "Naive Bayes"]
accuracy  = [0.880, 0.785, 0.860]
precision = [0.785, 0.673, 0.811]
recall    = [0.836, 0.574, 0.705]
f1        = [0.810, 0.619, 0.754]

x = np.arange(len(models))
w = 0.2
plt.figure(figsize=(8, 5))
plt.bar(x - 1.5*w, accuracy, w, label="Accuracy")
plt.bar(x - 0.5*w, precision, w, label="Precision")
plt.bar(x + 0.5*w, recall, w, label="Recall")
plt.bar(x + 1.5*w, f1, w, label="F1 score")
plt.xticks(x, models)
plt.ylim(0, 1)
plt.title("Model comparison (after feature engineering)")
plt.legend()
plt.savefig("images/model_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_pred = log_model.predict(X_test_scaled)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Rejected", "Approved"])
plt.title("Confusion Matrix - Logistic Regression")
plt.savefig("images/confusion_matrix.png", dpi=100, bbox_inches="tight")
plt.show()